# Sensitivity to recorded output count

CPU only. Run AFTER notebook 02 has produced the saved pools and evaluations for all five seeds. Nested total sizes 250, 500, 1000; fixed detection budget. This is not a rerun of the main leave-one-family-out experiment.

In [ ]:
from pathlib import Path
import os
import json
from IPython.display import display

# Edit this ONE setting if your collections live elsewhere.
DATA_ROOT = Path(os.environ.get("B3IT_DATA_ROOT", "/content/drive/MyDrive"))
if not DATA_ROOT.exists():
    try:
        from google.colab import drive
        drive.mount("/content/drive")
    except ImportError:
        raise FileNotFoundError("Set B3IT_DATA_ROOT to the directory containing both model folders.")
assert DATA_ROOT.exists(), DATA_ROOT
collections = {
    "Qwen": DATA_ROOT / "B3IT_Qwen05_Noise_Pruning" / "expanded_scales_1000samples_v1",
    "Phi": DATA_ROOT / "B3IT_Phi4mini_Noise_Pruning" / "expanded_scales_1000samples_v1",
}
folders = {model: folder / "extension_selection_repeated_v1" for model, folder in collections.items()}


## Freeze/check sensitivity protocol

In [ ]:
from pathlib import Path
from itertools import combinations
import hashlib
import json

import numpy as np
import pandas as pd
import scipy
from scipy.stats import rankdata
from IPython.display import display

BASE = DATA_ROOT

assert BASE.exists(), "Monte Google Drive avant cette cellule."

SIZE_COLLECTIONS = {
    "Qwen": BASE / "B3IT_Qwen05_Noise_Pruning"
                 / "expanded_scales_1000samples_v1",
    "Phi": BASE / "B3IT_Phi4mini_Noise_Pruning"
                / "expanded_scales_1000samples_v1",
}

SIZE_SEEDS = [101, 202, 303, 404, 505]
SIZE_LEVELS = [250, 500, 1000]
SIZE_TESTS = 1000
SIZE_REF = 50
SIZE_DET = 10


def size_digest(value):
    text = json.dumps(
        value, sort_keys=True, ensure_ascii=False,
        separators=(",", ":")
    )
    return hashlib.sha256(text.encode()).hexdigest()


def size_rng(model, seed, operation, *identifiers):
    # Same deterministic RNG convention as the single-family experiment.
    key = json.dumps(
        [model, seed, operation, *identifiers],
        ensure_ascii=False, separators=(",", ":")
    ).encode()
    return np.random.default_rng(
        int.from_bytes(hashlib.sha256(key).digest(), "big")
    )


def size_save(path, value):
    temporary = path.with_suffix(".tmp")
    temporary.write_text(
        json.dumps(value, ensure_ascii=False, indent=2)
    )
    temporary.replace(path)


SIZE_PROTOCOLS = {}

for label, folder in SIZE_COLLECTIONS.items():
    source = folder / "extension_single_family_v1"
    original = json.loads((source / "protocol.json").read_text())

    required = [
        source / f"{prefix}_seed_{seed}.json"
        for seed in SIZE_SEEDS
        for prefix in ["pools", "evaluation"]
    ]
    for path in required:
        assert path.exists(), f"Fichier absent : {path}"

    protocol = {
        "experiment": "recorded_output_size_sensitivity_v1",
        "model": original["model"],
        "seeds": SIZE_SEEDS,
        "recorded_outputs_per_input_version": SIZE_LEVELS,
        "pool_sizes": {"250": 125, "500": 250, "1000": 500},
        "candidate_prompts": original["candidate_prompts"],
        "families": original["families"],
        "train_test_pairs": original["train_test_pairs"],
        "selected_inputs": 3,
        "reference_samples_per_input": SIZE_REF,
        "detection_samples_per_input": SIZE_DET,
        "simulation_tests_per_class": SIZE_TESTS,
        "design": (
            "Reuse saved disjoint 500/500 pools. Separately thin each "
            "pool without replacement: 500 -> 250 -> 125. "
            "Recompute selection and evaluation at every size."
        ),
        "selection": (
            "Single-family mean individual-input training AUC. "
            "Top three; ties follow original candidate order."
        ),
        "random_baseline": "Mean AUC over all 120 triples",
        "simulation": (
            "Multinomial sampling with replacement within each pool. "
            "Same RNG identifiers across sizes."
        ),
        "interpretation": (
            "Post-hoc sensitivity analysis of one recorded collection. "
            "Seed SD is not a confidence interval. "
            "No conclusion about collections larger than 1000."
        ),
        "numpy_version": np.__version__,
        "scipy_version": scipy.__version__,
        "source_file_sha256": {
            p.name: hashlib.sha256(p.read_bytes()).hexdigest()
            for p in required
        },
    }

    output = folder / "extension_output_size_sensitivity_v1"
    output.mkdir(exist_ok=True)
    path = output / "protocol.json"

    if path.exists():
        assert json.loads(path.read_text()) == protocol, (
            "Protocole différent déjà présent. "
            "Utilise un nouveau nom de dossier."
        )
    else:
        size_save(path, protocol)

    SIZE_PROTOCOLS[label] = protocol
    print(f"{label} : protocole sauvegardé dans {output}")

print("\n250 sorties = 125 sélection + 125 évaluation")
print("500 sorties = 250 sélection + 250 évaluation")
print("1000 sorties = 500 sélection + 500 évaluation")
print("Budget de détection inchangé : 3 × 10 = 30")

## Sampling and simulation helpers

In [ ]:
def size_thin(counts, number, rng):
    """Sample recorded observations without replacement."""
    tokens = sorted(counts)
    values = np.array([counts[t] for t in tokens], dtype=np.int64)
    assert values.sum() >= number

    drawn = rng.multivariate_hypergeometric(values, number)
    result = {t: int(n) for t, n in zip(tokens, drawn)}

    assert sum(result.values()) == number
    assert all(result[t] <= counts[t] for t in tokens)
    return result


def size_nested_pools(saved):
    """Keep selection/evaluation separate at every size."""
    levels = {1000: {
        "selection": saved["selection"],
        "evaluation": saved["evaluation"],
    }}

    for total, parent in [(500, 1000), (250, 500)]:
        levels[total] = {}

        for role in ["selection", "evaluation"]:
            levels[total][role] = {}

            for variant, inputs in levels[parent][role].items():
                levels[total][role][variant] = {}

                for prompt, counts in inputs.items():
                    levels[total][role][variant][prompt] = size_thin(
                        counts, total // 2,
                        size_rng(
                            saved["protocol"]["model"], saved["seed"],
                            "output_size_thinning",
                            role, variant, prompt, total
                        )
                    )

    return levels


def size_tv(p, q, rng):
    reference = rng.multinomial(
        SIZE_REF, p, size=SIZE_TESTS
    ) / SIZE_REF

    detection = rng.multinomial(
        SIZE_DET, q, size=SIZE_TESTS
    ) / SIZE_DET

    return np.abs(reference - detection).sum(axis=1) / 2


def size_auc(null, changed):
    """AUC for one or several columns, with half credit for ties."""
    n0, n1 = len(null), len(changed)
    ranks = rankdata(
        np.concatenate([null, changed], axis=0),
        axis=0, method="average"
    )
    return (
        ranks[n0:].sum(axis=0) - n1 * (n1 + 1) / 2
    ) / (n0 * n1)


def size_score_matrices(counts, model, seed, role, prompts, variants):
    null = np.empty((SIZE_TESTS, len(prompts)))
    changed = {
        name: np.empty_like(null) for name in variants
    }

    for column, prompt in enumerate(prompts):
        tokens = sorted({
            token
            for version in counts.values()
            for token in version[prompt]
        })

        def probabilities(name):
            values = np.array([
                counts[name][prompt].get(t, 0) for t in tokens
            ], dtype=float)
            return values / values.sum()

        baseline = probabilities("unchanged")

        null[:, column] = size_tv(
            baseline, baseline,
            size_rng(model, seed, f"{role}_null", prompt)
        )

        for name in variants:
            changed[name][:, column] = size_tv(
                baseline, probabilities(name),
                size_rng(
                    model, seed, f"{role}_changed", name, prompt
                )
            )

    return null, changed


print("Fonctions prêtes.")

## Recompute all sizes; check the 1000-output endpoint

In [ ]:
size_rows = []

for label, folder in SIZE_COLLECTIONS.items():
    protocol = SIZE_PROTOCOLS[label]
    model = protocol["model"]
    prompts = protocol["candidate_prompts"]
    families = protocol["families"]

    variants = [
        name for names in families.values() for name in names
    ]

    triples = np.array(list(combinations(range(len(prompts)), 3)))
    assert len(triples) == 120
    lookup = {
        frozenset(indices): i
        for i, indices in enumerate(triples)
    }

    source = folder / "extension_single_family_v1"
    output = folder / "extension_output_size_sensitivity_v1"

    for seed in SIZE_SEEDS:
        saved = json.loads(
            (source / f"pools_seed_{seed}.json").read_text()
        )
        assert saved["protocol"]["candidate_prompts"] == prompts
        assert saved["protocol"]["families"] == families

        levels = size_nested_pools(saved)

        for total in SIZE_LEVELS:
            path = output / f"seed_{seed}_outputs_{total}.json"

            if path.exists():
                cached = json.loads(path.read_text())
                assert cached["protocol_sha256"] == size_digest(protocol)
                size_rows.extend(cached["results"])
                print(f"{label} / {seed} / {total} : rechargé",
                      flush=True)
                continue

            print(f"{label} / {seed} / {total} : calcul...",
                  flush=True)

            pools = levels[total]

            # Rank individual inputs using selection data only.
            null_s, changed_s = size_score_matrices(
                pools["selection"], model, seed,
                "selection", prompts, variants
            )

            input_aucs = {
                name: size_auc(null_s, changed_s[name])
                for name in variants
            }

            choices = {}
            for family, names in families.items():
                quality = np.mean(
                    [input_aucs[name] for name in names], axis=0
                )
                choices[family] = np.argsort(
                    -quality, kind="stable"
                )[:3].tolist()

            entropies = []
            for prompt in prompts:
                values = np.array(list(
                    pools["selection"]["unchanged"][prompt].values()
                ), dtype=float)
                probabilities = values[values > 0] / values.sum()
                entropies.append(
                    -(probabilities * np.log2(probabilities)).sum()
                )

            entropy_choice = np.argsort(
                -np.array(entropies), kind="stable"
            )[:3]
            entropy_index = lookup[frozenset(entropy_choice)]

            # Evaluate all 120 triples on reserved data.
            null_e, changed_e = size_score_matrices(
                pools["evaluation"], model, seed,
                "evaluation", prompts, variants
            )
            null_groups = null_e[:, triples].mean(axis=2)

            auc_by_variant = {
                name: size_auc(
                    null_groups,
                    changed_e[name][:, triples].mean(axis=2)
                )
                for name in variants
            }

            # At 1000, verify the original single-family evaluation.
            if total == 1000:
                previous = json.loads(
                    (source / f"evaluation_seed_{seed}.json").read_text()
                )
                previous_lookup = {
                    frozenset(group): i
                    for i, group in enumerate(previous["subsets"])
                }
                order = [
                    previous_lookup[
                        frozenset(prompts[i] for i in triple)
                    ]
                    for triple in triples
                ]

                for name in variants:
                    expected = np.array(
                        previous["auc_by_variant"][name]
                    )[order]
                    assert np.allclose(
                        auc_by_variant[name], expected,
                        atol=1e-12, rtol=0
                    ), (
                        "Le point 1000 ne reproduit pas l'évaluation "
                        "précédente. Vérifie les versions et les pools."
                    )

            rows = []
            for pair in protocol["train_test_pairs"]:
                train, test = pair["train"], pair["test"]
                chosen = choices[train]
                chosen_index = lookup[frozenset(chosen)]

                family_aucs = np.mean([
                    auc_by_variant[name] for name in families[test]
                ], axis=0)

                performance = float(family_aucs[chosen_index])
                entropy = float(family_aucs[entropy_index])
                random_mean = float(family_aucs.mean())

                rows.append({
                    "model": label,
                    "seed": seed,
                    "n_outputs": total,
                    "training_family": train,
                    "test_family": test,
                    "performance_auc": performance,
                    "entropy_auc": entropy,
                    "random_mean_auc": random_mean,
                    "gain_vs_random": performance - random_mean,
                    "gain_vs_entropy": performance - entropy,
                    "selected_prompts": [prompts[i] for i in chosen],
                })

            size_save(path, {
                "protocol_sha256": size_digest(protocol),
                "results": rows,
                "selected_by_family": {
                    family: [prompts[i] for i in indices]
                    for family, indices in choices.items()
                },
            })
            size_rows.extend(rows)
            print("  Sauvegardé.", flush=True)

size_results = pd.DataFrame(size_rows)

size_summary = (
    size_results.groupby([
        "model", "training_family", "test_family", "n_outputs"
    ])
    .agg(
        performance_mean=("performance_auc", "mean"),
        performance_sd=("performance_auc", "std"),
        entropy_mean=("entropy_auc", "mean"),
        random_mean=("random_mean_auc", "mean"),
        gain_vs_random=("gain_vs_random", "mean"),
        gain_sd=("gain_vs_random", "std"),
        positive_vs_random=(
            "gain_vs_random", lambda x: int((x > 0).sum())
        ),
    )
    .reset_index()
)

for label, folder in SIZE_COLLECTIONS.items():
    output = folder / "extension_output_size_sensitivity_v1"
    size_results[size_results.model == label].to_csv(
        output / "all_seeds.csv", index=False
    )
    size_summary[size_summary.model == label].to_csv(
        output / "summary.csv", index=False
    )

display(size_summary.round(4))
print("Terminé. SD = variation entre graines, pas un IC.")

## Summaries and sensitivity plots

In [ ]:
import matplotlib.pyplot as plt

family_labels = {
    "magnitude": "Magnitude pruning",
    "random": "Random pruning",
    "random_noise": "Gaussian noise",
}

for label, folder in SIZE_COLLECTIONS.items():
    data = size_summary[size_summary.model == label]
    pairs = SIZE_PROTOCOLS[label]["train_test_pairs"]

    fig, axes = plt.subplots(
        2, 3, figsize=(14, 7), sharex=True
    )

    for ax, pair in zip(axes.flat, pairs):
        train, test = pair["train"], pair["test"]
        part = data[
            (data.training_family == train)
            & (data.test_family == test)
        ].sort_values("n_outputs")

        ax.errorbar(
            part.n_outputs, part.performance_mean,
            yerr=part.performance_sd,
            marker="o", capsize=3, label="Performance-based"
        )
        ax.plot(
            part.n_outputs, part.entropy_mean,
            marker="s", label="Entropy-based"
        )
        ax.plot(
            part.n_outputs, part.random_mean,
            marker="^", label="Random mean"
        )

        ax.set_title(
            f"Select: {family_labels[train]}\n"
            f"Test: {family_labels[test]}"
        )
        ax.set_xticks(SIZE_LEVELS)
        ax.set_xlabel("Recorded outputs per input/version")
        ax.set_ylabel("Held-out ROC AUC")
        ax.grid(alpha=0.25)

    handles, labels = axes.flat[0].get_legend_handles_labels()
    fig.legend(handles, labels, loc="upper center", ncol=3)
    fig.suptitle(
        f"{label}: sensitivity to recorded output count",
        y=1.04
    )
    fig.tight_layout(rect=[0, 0, 1, 0.94])

    output = folder / "extension_output_size_sensitivity_v1"
    fig.savefig(output / "output_size_auc.png",
                dpi=200, bbox_inches="tight")
    fig.savefig(output / "output_size_auc.pdf",
                bbox_inches="tight")
    plt.show()

# Paired differences: same model, seed and train/test direction.
comparison = size_results.pivot(
    index=["model", "seed", "training_family", "test_family"],
    columns="n_outputs",
    values=["performance_auc", "gain_vs_random"]
)

changes = pd.DataFrame({
    "auc_change_1000_minus_500":
        comparison["performance_auc"][1000]
        - comparison["performance_auc"][500],
    "gain_change_1000_minus_500":
        comparison["gain_vs_random"][1000]
        - comparison["gain_vs_random"][500],
}).reset_index()

change_summary = (
    changes.groupby(["model", "training_family", "test_family"])
    .agg(
        mean_auc_change=("auc_change_1000_minus_500", "mean"),
        mean_absolute_auc_change=(
            "auc_change_1000_minus_500",
            lambda x: x.abs().mean()
        ),
        mean_gain_change=("gain_change_1000_minus_500", "mean"),
    )
    .reset_index()
)

display(change_summary.round(4))

for label, folder in SIZE_COLLECTIONS.items():
    change_summary[change_summary.model == label].to_csv(
        folder / "extension_output_size_sensitivity_v1"
        / "changes_500_to_1000.csv",
        index=False
    )

print(
    "Barres : ±1 SD entre cinq graines pour performance-based.\n"
    "Les tailles réutilisent des données imbriquées.\n"
    "La stabilité éventuelle reste conditionnelle à cette collection."
)